# Web-Based Healthcare Analytics and Patient Readmission Risk Prediction

**Subject:** Graph and Web Analytics  
**Platform:** Google Colab  
**Language:** Python  

### Project goal
Analyze hospital patient records, study readmission patterns, represent patient–diagnosis relationships as a graph, extract simple network features, predict readmission risk, and present the results through a lightweight Streamlit web dashboard.

> **Dataset:** Diabetes 130-US Hospitals for Years 1999–2008 from the UCI Machine Learning Repository.

## 1. Project Architecture

**Dataset → Cleaning → Healthcare Analytics → Patient–Diagnosis Graph → Graph Metrics → Machine Learning → Web Dashboard**

### Simplified scope
- One main healthcare graph: **Patient–Diagnosis**
- Three graph metrics: **Degree Centrality, Betweenness Centrality, PageRank**
- Two ML models: **Logistic Regression and Random Forest**
- Professional charts using Matplotlib/Seaborn
- Streamlit dashboard with three views: Analytics, Graph Analytics, Prediction

In [ ]:
# Install required packages
!pip -q install ucimlrepo networkx scikit-learn seaborn joblib streamlit pyngrok

In [ ]:
# Imports
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import joblib

from itertools import combinations
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

# Professional, presentation-friendly defaults without hard-coded chart colors
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titleweight": "bold",
    "axes.labelweight": "semibold",
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "savefig.bbox": "tight"
})

OUTPUT_DIR = "project_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load the UCI Dataset

The notebook fetches the official UCI dataset directly, so you do not need to manually place the CSV in Colab.

In [ ]:
# Fetch official UCI dataset
dataset = fetch_ucirepo(id=296)

df = dataset.data.original.copy()

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Basic dataset information
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing": df.isna().sum().values
}).sort_values("missing", ascending=False).head(15))

## 3. Data Cleaning

For this simplified academic project, we remove identifier/high-missingness fields and convert the original `readmitted` target into a binary readmission indicator.

In [ ]:
# Replace '?' with missing values
df = df.replace("?", np.nan)

# Remove identifier / high-missingness columns that are not needed for this project
drop_cols = [
    "encounter_id", "patient_nbr",
    "weight", "payer_code", "medical_specialty"
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Binary target:
# NO  -> 0 (not readmitted)
# <30 or >30 -> 1 (readmitted)
df["readmission"] = df["readmitted"].apply(lambda x: 0 if x == "NO" else 1)
df = df.drop(columns=["readmitted"])

print("Cleaned shape:", df.shape)
display(df.head())

In [ ]:
# Target distribution
target_counts = df["readmission"].value_counts().sort_index()
target_labels = ["Not Readmitted", "Readmitted"]

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    x=target_labels,
    y=[target_counts.get(0, 0), target_counts.get(1, 0)],
    ax=ax
)
ax.set_title("Patient Readmission Distribution")
ax.set_xlabel("")
ax.set_ylabel("Number of encounters")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_readmission_distribution.png")
plt.show()

## 4. Healthcare Analytics

In [ ]:
# Professional chart 1: readmission rate by age group
age_rate = (
    df.groupby("age", observed=True)["readmission"]
      .mean()
      .mul(100)
      .reset_index(name="readmission_rate")
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=age_rate, x="age", y="readmission_rate", ax=ax)
ax.set_title("Readmission Rate by Age Group")
ax.set_xlabel("Age group")
ax.set_ylabel("Readmission rate (%)")
ax.tick_params(axis="x", rotation=35)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=2, fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_readmission_by_age.png")
plt.show()

In [ ]:
# Professional chart 2: previous inpatient visits vs readmission
plot_df = (
    df.groupby("number_inpatient", dropna=False)["readmission"]
      .mean()
      .mul(100)
      .reset_index(name="readmission_rate")
)
plot_df["number_inpatient"] = plot_df["number_inpatient"].astype(int)
plot_df = plot_df[plot_df["number_inpatient"] <= 8]

fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(
    data=plot_df,
    x="number_inpatient",
    y="readmission_rate",
    marker="o",
    linewidth=2.5,
    ax=ax
)
ax.set_title("Readmission Rate vs Previous Inpatient Visits")
ax.set_xlabel("Previous inpatient visits")
ax.set_ylabel("Readmission rate (%)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_inpatient_vs_readmission.png")
plt.show()

In [ ]:
# Professional chart 3: hospital stay vs readmission
stay_df = (
    df.groupby("time_in_hospital")["readmission"]
      .mean()
      .mul(100)
      .reset_index(name="readmission_rate")
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(
    data=stay_df,
    x="time_in_hospital",
    y="readmission_rate",
    marker="o",
    linewidth=2.5,
    ax=ax
)
ax.set_title("Readmission Rate vs Length of Hospital Stay")
ax.set_xlabel("Days in hospital")
ax.set_ylabel("Readmission rate (%)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_stay_vs_readmission.png")
plt.show()

## 5. Healthcare Graph: Patient–Diagnosis Network

To keep the project manageable, we construct one main graph using a representative sample of encounters. Each patient is connected to their primary, secondary, and tertiary diagnoses.

In [ ]:
# Build a manageable patient-diagnosis graph from a reproducible sample
graph_df = df.sample(n=min(5000, len(df)), random_state=42).copy()

G = nx.Graph()

for idx, row in graph_df.iterrows():
    patient = f"P_{idx}"

    diagnoses = [row.get("diag_1"), row.get("diag_2"), row.get("diag_3")]
    diagnoses = [d for d in diagnoses if pd.notna(d)]

    if not diagnoses:
        continue

    G.add_node(patient, node_type="patient")

    for diagnosis in set(diagnoses):
        diagnosis_node = f"D_{diagnosis}"
        G.add_node(diagnosis_node, node_type="diagnosis")
        G.add_edge(patient, diagnosis_node)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

In [ ]:
# Professional network visualization
plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=0.35, iterations=40)

patient_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "patient"]
diagnosis_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "diagnosis"]

nx.draw_networkx_edges(G, pos, alpha=0.18, width=0.7)
nx.draw_networkx_nodes(G, pos, nodelist=patient_nodes, node_size=8, alpha=0.55)
nx.draw_networkx_nodes(G, pos, nodelist=diagnosis_nodes, node_size=120, alpha=0.9)

top_degree_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:12]
labels = {n: n.replace("D_", "") for n, _ in top_degree_nodes if n.startswith("D_")}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)

plt.title("Patient–Diagnosis Healthcare Network")
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_patient_diagnosis_network.png")
plt.show()

## 6. Graph Analytics

In [ ]:
# Calculate three simple graph metrics
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G, k=min(300, len(G)), seed=42)
pagerank = nx.pagerank(G)

diagnosis_metrics = pd.DataFrame({
    "node": list(pagerank.keys()),
    "degree_centrality": [degree_centrality[n] for n in pagerank],
    "betweenness": [betweenness_centrality[n] for n in pagerank],
    "pagerank": [pagerank[n] for n in pagerank]
})

diagnosis_metrics = diagnosis_metrics[
    diagnosis_metrics["node"].str.startswith("D_")
].copy()

diagnosis_metrics["diagnosis"] = diagnosis_metrics["node"].str.replace("D_", "", regex=False)

display(
    diagnosis_metrics.sort_values("pagerank", ascending=False)
    .head(10)
    [["diagnosis", "degree_centrality", "betweenness", "pagerank"]]
)

In [ ]:
# Professional chart: top diagnoses by PageRank
top_pr = (
    diagnosis_metrics.sort_values("pagerank", ascending=False)
    .head(10)
    .sort_values("pagerank")
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(
    data=top_pr,
    x="pagerank",
    y="diagnosis",
    ax=ax
)
ax.set_title("Top Diagnoses by PageRank")
ax.set_xlabel("PageRank")
ax.set_ylabel("Diagnosis code")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/06_top_diagnoses_pagerank.png")
plt.show()

## 7. Machine Learning Readmission Prediction

The ML model uses a small, explainable feature set. A Random Forest and Logistic Regression model are compared.

In [ ]:
# Select simple, explainable predictors
model_features = [
    "age",
    "gender",
    "time_in_hospital",
    "num_medications",
    "number_diagnoses",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

model_df = df[model_features + ["readmission"]].copy()

X = model_df[model_features]
y = model_df["readmission"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

categorical_features = ["age", "gender"]
numeric_features = [
    "time_in_hospital",
    "num_medications",
    "number_diagnoses",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

print("Models trained successfully.")

In [ ]:
# Model evaluation
def evaluate_model(name, model):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1 Score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    }

results = pd.DataFrame([
    evaluate_model("Logistic Regression", logistic_model),
    evaluate_model("Random Forest", random_forest_model)
])

display(results.round(4))
results.to_csv(f"{OUTPUT_DIR}/model_comparison.csv", index=False)

In [ ]:
# Professional model comparison chart
metrics_long = results.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    var_name="Metric",
    value_name="Score"
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=metrics_long, x="Metric", y="Score", hue="Model", ax=ax)
ax.set_title("Model Performance Comparison")
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend(title="")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/07_model_comparison.png")
plt.show()

In [ ]:
# Confusion matrix for Random Forest
rf_pred = random_forest_model.predict(X_test)

cm = confusion_matrix(y_test, rf_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    square=True,
    cbar=False,
    xticklabels=["Not Readmitted", "Readmitted"],
    yticklabels=["Not Readmitted", "Readmitted"],
    ax=ax
)
ax.set_title("Random Forest Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/08_confusion_matrix.png")
plt.show()

In [ ]:
# ROC curve
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_estimator(
    logistic_model, X_test, y_test, name="Logistic Regression", ax=ax
)
RocCurveDisplay.from_estimator(
    random_forest_model, X_test, y_test, name="Random Forest", ax=ax
)
ax.set_title("ROC Curves for Readmission Prediction")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/09_roc_curve.png")
plt.show()

## 8. Save the Trained Model

The saved model can be loaded by the Streamlit dashboard.

In [ ]:
# Save the Random Forest pipeline for the dashboard
joblib.dump(random_forest_model, f"{OUTPUT_DIR}/readmission_model.joblib")

# Save a compact sample for the dashboard/demo
demo_columns = model_features + ["readmission"]
model_df.head(5000).to_csv(
    f"{OUTPUT_DIR}/demo_patient_data.csv",
    index=False
)

print("Saved model and demo data to:", OUTPUT_DIR)

## 9. Final Project Summary

### What the system demonstrates
1. Hospital data cleaning and exploratory analysis.
2. Readmission pattern analysis.
3. Patient–diagnosis graph construction.
4. Degree centrality, betweenness centrality, and PageRank.
5. Readmission prediction using Logistic Regression and Random Forest.
6. Professional visual analytics.
7. A Streamlit web dashboard using the saved Random Forest pipeline.

### Academic value
The project combines **healthcare analytics + graph analytics + machine learning + web analytics** in one manageable implementation.

In [ ]:
# Quick final summary
print("FINAL PROJECT SUMMARY")
print("-" * 60)
print(f"Dataset rows used after cleaning: {len(df):,}")
print(f"Graph nodes: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")
print("\nModel results:")
display(results.round(4))


## 10. Streamlit Web Dashboard — Added Section

This section adds the Streamlit web interface without changing any of the existing project analysis.

The dashboard contains:
1. Healthcare Analytics
2. Graph Analytics
3. New Patient Readmission Detection

The existing trained Random Forest model and existing `project_outputs` files are reused.

In [ ]:
# Streamlit + ngrok setup — updated requested cell only
# Enter your own ngrok authentication token below.
# Do not share your token publicly.

NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

from pyngrok import ngrok
import os
import subprocess
import time

if NGROK_AUTH_TOKEN == "PASTE_YOUR_NGROK_TOKEN_HERE":
    print("Please replace PASTE_YOUR_NGROK_TOKEN_HERE with your ngrok token and run this cell again.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Start Streamlit in the background.
    streamlit_process = subprocess.Popen(
        [
            "streamlit", "run", "app.py",
            "--server.port", "8501",
            "--server.address", "0.0.0.0",
            "--server.headless", "true"
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT
    )

    time.sleep(5)

    # Create a public ngrok tunnel to the Streamlit port.
    public_url = ngrok.connect(8501, "http").public_url

    print("✅ Streamlit is running.")
    print("🌐 OPEN THIS LINK TO VIEW THE WEBPAGE:")
    print(public_url)
    print("\nKeep this Colab session running while using the webpage.")



## 11. New Patient Readmission Detection — Added Cell

This section uses the **existing trained Random Forest model** to detect readmission risk for a new patient.

A new patient's details are entered using the same features used during training. The model returns:
- Readmission / Not Readmitted
- Estimated readmission probability

This does not retrain or change the existing model.

In [ ]:
# New Patient Readmission Detection — added cell only
saved_model = joblib.load(f"{OUTPUT_DIR}/readmission_model.joblib")

# Example NEW patient
new_patient = pd.DataFrame([{
    "age": "[70-80)",
    "gender": "Female",
    "time_in_hospital": 6,
    "num_medications": 20,
    "number_diagnoses": 8,
    "number_outpatient": 1,
    "number_emergency": 1,
    "number_inpatient": 2
}])

prediction = saved_model.predict(new_patient)[0]
probability = saved_model.predict_proba(new_patient)[0, 1]

print("NEW PATIENT READMISSION DETECTION")
print("=" * 40)
display(new_patient)
print("Prediction:", "Readmission" if prediction == 1 else "Not Readmitted")
print(f"Estimated readmission probability: {probability:.2%}")


## 12. Streamlit App Code — Added Cell

The following cell creates `app.py`. It only uses the model, charts, and data already produced by the existing notebook.

In [ ]:
# Create the Streamlit dashboard — added cell only
streamlit_code = r'''import os
import joblib
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt

st.set_page_config(page_title="Healthcare Analytics Dashboard", page_icon="🏥", layout="wide")
OUTPUT_DIR = "project_outputs"
MODEL_PATH = os.path.join(OUTPUT_DIR, "readmission_model.joblib")
DEMO_PATH = os.path.join(OUTPUT_DIR, "demo_patient_data.csv")

st.title("🏥 Web-Based Healthcare Analytics")
st.caption("Healthcare Analytics • Graph Analytics • New Patient Readmission Detection")

if not os.path.exists(MODEL_PATH):
    st.error("Run the Colab notebook first so the saved Random Forest model is available.")
    st.stop()

model = joblib.load(MODEL_PATH)
demo = pd.read_csv(DEMO_PATH) if os.path.exists(DEMO_PATH) else pd.DataFrame()

page = st.sidebar.radio("Navigation", ["Healthcare Analytics", "Graph Analytics", "New Patient Readmission Detection"])

if page == "Healthcare Analytics":
    st.header("📊 Healthcare Analytics")
    if demo.empty:
        st.warning("Run the notebook first to generate the dashboard data.")
        st.stop()
    tabs = st.tabs(["Age Analysis", "Hospital Stay", "Previous Visits"])
    with tabs[0]:
        age_rate = demo.groupby("age")["readmission"].mean().mul(100)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(age_rate.index, age_rate.values)
        ax.set_title("Readmission Rate by Age Group")
        ax.set_ylabel("Readmission Rate (%)")
        ax.tick_params(axis="x", rotation=45)
        fig.tight_layout()
        st.pyplot(fig)
    with tabs[1]:
        stay_rate = demo.groupby("time_in_hospital")["readmission"].mean().mul(100)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(stay_rate.index, stay_rate.values, marker="o")
        ax.set_title("Readmission Rate vs Hospital Stay")
        ax.set_xlabel("Days in Hospital")
        ax.set_ylabel("Readmission Rate (%)")
        fig.tight_layout()
        st.pyplot(fig)
    with tabs[2]:
        visit_rate = demo.groupby("number_inpatient")["readmission"].mean().mul(100)
        visit_rate = visit_rate[visit_rate.index <= 8]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(visit_rate.index, visit_rate.values, marker="o")
        ax.set_title("Readmission Rate vs Previous Inpatient Visits")
        ax.set_xlabel("Previous Inpatient Visits")
        ax.set_ylabel("Readmission Rate (%)")
        fig.tight_layout()
        st.pyplot(fig)

elif page == "Graph Analytics":
    st.header("🕸️ Graph Analytics")
    st.write("Patient–Diagnosis Network with Degree Centrality, Betweenness Centrality, and PageRank.")
    graph_path = os.path.join(OUTPUT_DIR, "05_patient_diagnosis_network.png")
    pagerank_path = os.path.join(OUTPUT_DIR, "06_top_diagnoses_pagerank.png")
    if os.path.exists(graph_path):
        st.image(graph_path, caption="Patient–Diagnosis Network", use_container_width=True)
    if os.path.exists(pagerank_path):
        st.image(pagerank_path, caption="Top Diagnoses by PageRank", use_container_width=True)
    c1, c2, c3 = st.columns(3)
    c1.metric("Degree Centrality", "Connections")
    c2.metric("Betweenness Centrality", "Bridge role")
    c3.metric("PageRank", "Network importance")

else:
    st.header("🧑‍⚕️ New Patient Readmission Detection")
    st.write("Enter details for a new patient and click the button to obtain a model prediction.")
    c1, c2 = st.columns(2)
    with c1:
        age = st.selectbox("Age Group", ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)", "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"], index=7)
        gender = st.selectbox("Gender", ["Male", "Female"])
        time_in_hospital = st.slider("Time in Hospital (days)", 1, 14, 5)
        num_medications = st.slider("Number of Medications", 0, 80, 15)
    with c2:
        number_diagnoses = st.slider("Number of Diagnoses", 1, 16, 6)
        number_outpatient = st.slider("Previous Outpatient Visits", 0, 10, 0)
        number_emergency = st.slider("Previous Emergency Visits", 0, 10, 0)
        number_inpatient = st.slider("Previous Inpatient Visits", 0, 10, 0)

    if st.button("🔍 Detect Readmission Risk", type="primary"):
        new_patient = pd.DataFrame([{
            "age": age, "gender": gender, "time_in_hospital": time_in_hospital,
            "num_medications": num_medications, "number_diagnoses": number_diagnoses,
            "number_outpatient": number_outpatient, "number_emergency": number_emergency,
            "number_inpatient": number_inpatient
        }])
        prediction = model.predict(new_patient)[0]
        probability = model.predict_proba(new_patient)[0, 1]
        st.subheader("Prediction Result")
        if prediction == 1:
            st.error("⚠️ Predicted: Readmission")
        else:
            st.success("✅ Predicted: Not Readmitted")
        st.metric("Estimated Readmission Probability", f"{probability:.2%}")
        st.progress(float(probability))
        st.info("Academic project demonstration only. This is not a clinical diagnosis.")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print("app.py created successfully.")
print("Run with: streamlit run app.py")